In [0]:
# Import required libraries
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.clustering import KMeans
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import matplotlib.pyplot as plt
import seaborn as sns

# Load data from Databricks Catalog
df = spark.table("workspace.default.ott_combined_1000")

# Cache for performance

print(f"Dataset loaded: {df.count()} rows, {len(df.columns)} columns")
display(df.limit(5))

Dataset loaded: 1000 rows, 26 columns


UserID,Age,Gender,Region,Preferred_Language,Subscription_Plan,Signup_Date,Monthly_Spend_USD,Total_Watch_Hours_90,Num_Sessions_Last_30,Avg_Session_Min,Favorite_Genre,Preferred_Device,Num_Ratings,Avg_Rating,Reviews_Count,Sentiment_Score,Avg_Ad_View_Sec,Ad_Click_Rate,Implicit_Pref_Score,Top_Content_1,Top_Content_2,Top_Content_3,Last_Active_Date,Churn,Liked_Genres
200001,28,Female,India,English,Premium,2021-05-12,12.99,44.47,3,44.59,Thriller,Mobile,4,3.28,0,0.25,5.72,0.031,11.81,5606,5210,5517,2025-09-03,1,Thriller;Horror
200002,22,Female,India,Hindi,Basic,2021-08-30,3.99,23.77,6,41.1,Drama,Tablet,3,3.97,2,1.0,1.0,0.09,3.23,5092,5704,5243,2025-11-11,0,Drama;Horror
200003,45,Male,India,French,Basic,2020-01-05,3.99,10.28,5,33.56,Drama,Mobile,6,2.95,2,1.0,11.62,0.02,2.01,5420,5300,5630,2025-11-05,0,Drama;Horror
200004,36,Female,USA,Hindi,Basic,2024-11-08,3.99,52.38,8,35.34,Thriller,Mobile,4,3.88,4,1.0,17.39,0.152,6.97,5081,5124,5172,2025-10-21,0,Thriller;Romance
200005,30,Female,USA,English,Standard,2023-01-08,7.99,4.64,3,45.89,Thriller,Mobile,5,4.12,4,-0.2,10.81,0.074,1.23,5341,5796,5655,2025-10-30,0,Thriller;Drama


In [0]:
# Fix date columns
from pyspark.sql.types import DateType

df = df.withColumn("Signup_Date", F.to_date(F.col("Signup_Date"))) \
       .withColumn("Last_Active_Date", F.to_date(F.col("Last_Active_Date")))

# Feature Engineering
df_fe = df.withColumn("Account_Age_Days", 
                      F.datediff(F.current_date(), F.col("Signup_Date"))) \
          .withColumn("Days_Since_Active", 
                      F.datediff(F.current_date(), F.col("Last_Active_Date"))) \
          .withColumn("Revenue_Per_Hour", 
                      F.when(F.col("Total_Watch_Hours_90") > 0, 
                             F.col("Monthly_Spend_USD") / F.col("Total_Watch_Hours_90")).otherwise(0)) \
          .withColumn("Engagement_Score", 
                      F.col("Total_Watch_Hours_90") * F.col("Num_Sessions_Last_30") / 100) \
          .withColumn("Activity_Ratio", 
                      F.col("Days_Since_Active") / F.when(F.col("Account_Age_Days") > 0, 
                                                          F.col("Account_Age_Days")).otherwise(1))

# Handle missing values
df_fe = df_fe.fillna({
    'Num_Ratings': 0,
    'Reviews_Count': 0,
    'Sentiment_Score': 0.5,
    'Ad_Click_Rate': 0
})

display(df_fe.select("UserID", "Account_Age_Days", "Revenue_Per_Hour", "Engagement_Score").limit(10))

UserID,Account_Age_Days,Revenue_Per_Hour,Engagement_Score
200001,1644,0.2921070384528896,1.3341
200002,1534,0.16785864535128314,1.4262000000000001
200003,2137,0.3881322957198444,0.514
200004,368,0.07617411225658649,4.1904
200005,1038,1.7219827586206897,0.1392
200006,200,0.08127928294968426,2.4545000000000003
200007,1117,0.05503448275862069,3.625
200008,1086,0.47250147841513895,0.5073000000000001
200009,1598,0.035378613229295974,6.766800000000001
200010,556,0.38138578978273635,3.406


In [0]:
# Age Groups Analysis
df_eda = df_fe.withColumn("Age_Group",
    F.when(F.col("Age") < 25, "18-24")
    .when((F.col("Age") >= 25) & (F.col("Age") < 35), "25-34")
    .when((F.col("Age") >= 35) & (F.col("Age") < 45), "35-44")
    .when((F.col("Age") >= 45) & (F.col("Age") < 55), "45-54")
    .otherwise("55+")
)

# Demographics Summary
demographics = df_eda.groupBy("Age_Group", "Gender").agg(
    F.count("*").alias("Users"),
    F.avg("Monthly_Spend_USD").alias("Avg_Spend"),
    F.avg("Total_Watch_Hours_90").alias("Avg_Watch_Hours"),
    F.avg("Churn").alias("Churn_Rate"),
    F.avg("Sentiment_Score").alias("Avg_Sentiment")
).orderBy("Age_Group", "Gender")

display(demographics)

Age_Group,Gender,Users,Avg_Spend,Avg_Watch_Hours,Churn_Rate,Avg_Sentiment
18-24,Female,185,5.246432432432438,52.39740540540538,0.07567567567567568,0.17848648648648638
18-24,Male,182,5.788626373626379,59.9692857142857,0.07692307692307693,0.13005494505494508
18-24,Other,5,7.19,57.919999999999995,0.0,-0.124
25-34,Female,173,6.488612716763012,58.22578034682083,0.09826589595375723,0.324335260115607
25-34,Male,164,5.973414634146348,57.72493902439024,0.08536585365853659,0.12841463414634147
25-34,Other,7,8.847142857142858,81.77714285714286,0.0,-0.07285714285714287
35-44,Female,65,6.28415384615385,58.37215384615386,0.046153846153846156,0.12153846153846154
35-44,Male,73,5.40315068493151,51.22917808219177,0.1232876712328767,0.20232876712328765
35-44,Other,5,4.792,88.354,0.2,0.30999999999999994
45-54,Female,62,5.621129032258068,43.5058064516129,0.08064516129032258,0.2058064516129032


In [0]:
# Subscription Analysis
subscription_metrics = df_fe.groupBy("Subscription_Plan").agg(
    F.count("*").alias("Total_Users"),
    F.sum("Monthly_Spend_USD").alias("Total_Revenue"),
    F.avg("Monthly_Spend_USD").alias("ARPU"),
    F.avg("Total_Watch_Hours_90").alias("Avg_Watch_Hours"),
    F.avg("Num_Sessions_Last_30").alias("Avg_Sessions"),
    F.avg("Churn").alias("Churn_Rate"),
    F.stddev("Monthly_Spend_USD").alias("Revenue_StdDev")
).orderBy(F.desc("Total_Revenue"))

display(subscription_metrics)

# Revenue Distribution
revenue_percentiles = df_fe.select(
    F.expr("percentile_approx(Monthly_Spend_USD, 0.25)").alias("Q1_Revenue"),
    F.expr("percentile_approx(Monthly_Spend_USD, 0.50)").alias("Median_Revenue"),
    F.expr("percentile_approx(Monthly_Spend_USD, 0.75)").alias("Q3_Revenue"),
    F.expr("percentile_approx(Monthly_Spend_USD, 0.95)").alias("P95_Revenue")
)
display(revenue_percentiles)

Subscription_Plan,Total_Users,Total_Revenue,ARPU,Avg_Watch_Hours,Avg_Sessions,Churn_Rate,Revenue_StdDev
Standard,357,2852.4299999999803,7.989999999999945,66.8526050420168,5.84593837535014,0.05322128851540616,0.0
Premium,128,1662.720000000001,12.990000000000007,80.12421874999998,6.1171875,0.0234375,0.0
Basic,324,1292.7600000000025,3.9900000000000078,50.34666666666666,6.095679012345679,0.08024691358024691,0.0
Free,191,0.0,0.0,33.739738219895294,4.141361256544503,0.20418848167539266,0.0


Q1_Revenue,Median_Revenue,Q3_Revenue,P95_Revenue
3.99,3.99,7.99,12.99


In [0]:
# Genre Analysis
genre_analysis = df_fe.groupBy("Favorite_Genre").agg(
    F.count("*").alias("Users"),
    F.avg("Total_Watch_Hours_90").alias("Avg_Watch_Hours"),
    F.avg("Avg_Rating").alias("Avg_Rating_Given"),
    F.avg("Sentiment_Score").alias("Avg_Sentiment"),
    F.avg("Churn").alias("Churn_Rate")
).orderBy(F.desc("Users"))

display(genre_analysis)

# Device Preference Analysis
device_metrics = df_fe.groupBy("Preferred_Device").agg(
    F.count("*").alias("Users"),
    F.avg("Avg_Session_Min").alias("Avg_Session_Duration"),
    F.avg("Ad_Click_Rate").alias("Avg_Ad_Click_Rate"),
    F.avg("Monthly_Spend_USD").alias("Avg_Spend")
).orderBy(F.desc("Users"))

display(device_metrics)

Favorite_Genre,Users,Avg_Watch_Hours,Avg_Rating_Given,Avg_Sentiment,Churn_Rate
Drama,258,62.83949612403101,3.821201550387599,0.16682170542635663,0.08527131782945736
Comedy,178,53.09876404494382,3.8560674157303376,0.22382022471910124,0.0898876404494382
Thriller,151,54.84675496688741,3.866754966887419,0.13986754966887416,0.0728476821192053
Action,119,55.85033613445376,3.787815126050421,0.14067226890756304,0.10084033613445378
Romance,116,57.9394827586207,3.7320689655172425,0.24801724137931044,0.11206896551724138
Documentary,70,59.30771428571426,3.7457142857142864,0.33399999999999996,0.04285714285714286
Horror,57,51.32824561403511,3.7833333333333328,0.09333333333333334,0.10526315789473684
Kids,51,48.79294117647058,3.7931372549019606,0.10058823529411764,0.0784313725490196


Preferred_Device,Users,Avg_Session_Duration,Avg_Ad_Click_Rate,Avg_Spend
Mobile,623,35.42629213483145,0.08053451043338693,5.796035313001533
Desktop,171,35.6,0.08314035087719299,5.9157894736842165
TV,166,35.72132530120482,0.07840963855421684,5.787409638554222
Tablet,40,32.798500000000004,0.08874999999999998,5.6167500000000015


In [0]:
# Cell 1: Setup and Data Loading
from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd

# Load your data
df = spark.table("workspace.default.ott_combined_1000")

# Basic data check
print(f"Total rows: {df.count()}")
print(f"Total columns: {len(df.columns)}")
df.printSchema()

Total rows: 1000
Total columns: 26
root
 |-- UserID: long (nullable = true)
 |-- Age: long (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Preferred_Language: string (nullable = true)
 |-- Subscription_Plan: string (nullable = true)
 |-- Signup_Date: date (nullable = true)
 |-- Monthly_Spend_USD: double (nullable = true)
 |-- Total_Watch_Hours_90: double (nullable = true)
 |-- Num_Sessions_Last_30: long (nullable = true)
 |-- Avg_Session_Min: double (nullable = true)
 |-- Favorite_Genre: string (nullable = true)
 |-- Preferred_Device: string (nullable = true)
 |-- Num_Ratings: long (nullable = true)
 |-- Avg_Rating: double (nullable = true)
 |-- Reviews_Count: long (nullable = true)
 |-- Sentiment_Score: double (nullable = true)
 |-- Avg_Ad_View_Sec: double (nullable = true)
 |-- Ad_Click_Rate: double (nullable = true)
 |-- Implicit_Pref_Score: double (nullable = true)
 |-- Top_Content_1: long (nullable = true)
 |-- Top_Content_2: long

In [0]:
# Cell 1: Setup and Data Preparation for Churn Prediction
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Load data
df = spark.table("workspace.default.ott_combined_1000")

# Select relevant features for churn prediction
feature_cols = [
    'Age', 'Monthly_Spend_USD', 'Total_Watch_Hours_90', 
    'Num_Sessions_Last_30', 'Avg_Session_Min', 'Num_Ratings',
    'Avg_Rating', 'Reviews_Count', 'Sentiment_Score', 
    'Ad_Click_Rate', 'Implicit_Pref_Score'
]

# Prepare data
df_churn = df.select(feature_cols + ['Churn', 'Gender', 'Subscription_Plan', 'Preferred_Device'])
df_churn = df_churn.filter(F.col('Churn').isNotNull())

# Handle missing values
for col in feature_cols:
    df_churn = df_churn.fillna({col: 0})

# Convert Churn to integer
df_churn = df_churn.withColumn('label', F.col('Churn').cast('integer'))

print(f"Dataset size: {df_churn.count()} rows")
print(f"Churn distribution:")
df_churn.groupBy('label').count().show()

Dataset size: 1000 rows
Churn distribution:
+-----+-----+
|label|count|
+-----+-----+
|    1|   87|
|    0|  913|
+-----+-----+



Use Case 1 Churn Prediction with Random Forest

In [0]:
# Cell 2: Feature Engineering and Random Forest Model
# String indexing for categorical variables
gender_indexer = StringIndexer(inputCol="Gender", outputCol="Gender_Index", handleInvalid="keep")
plan_indexer = StringIndexer(inputCol="Subscription_Plan", outputCol="Plan_Index", handleInvalid="keep")
device_indexer = StringIndexer(inputCol="Preferred_Device", outputCol="Device_Index", handleInvalid="keep")

# Combine all features
all_features = feature_cols + ['Gender_Index', 'Plan_Index', 'Device_Index']
assembler = VectorAssembler(inputCols=all_features, outputCol="features")

# Scale features
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")

# Random Forest Classifier
rf = RandomForestClassifier(
    featuresCol="scaled_features",
    labelCol="label",
    numTrees=100,
    maxDepth=10,
    seed=42
)

# Create pipeline
pipeline = Pipeline(stages=[
    gender_indexer, plan_indexer, device_indexer,
    assembler, scaler, rf
])

# Split data
train_data, test_data = df_churn.randomSplit([0.8, 0.2], seed=42)

# Train model
print("Training Random Forest model...")
rf_model = pipeline.fit(train_data)

# Make predictions
predictions = rf_model.transform(test_data)

# Evaluate model
print("\n=== RANDOM FOREST MODEL EVALUATION ===")

# Binary evaluator
binary_evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc = binary_evaluator.evaluate(predictions)
print(f"AUC-ROC: {auc:.4f}")

# Multiclass evaluator for accuracy
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
accuracy = accuracy_evaluator.evaluate(predictions)
print(f"Accuracy: {accuracy:.4f}")

# Precision evaluator
precision_evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision")
precision = precision_evaluator.evaluate(predictions)
print(f"Weighted Precision: {precision:.4f}")

# Recall evaluator
recall_evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall")
recall = recall_evaluator.evaluate(predictions)
print(f"Weighted Recall: {recall:.4f}")

# F1 Score
f1_evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
f1 = f1_evaluator.evaluate(predictions)
print(f"F1 Score: {f1:.4f}")

# Feature importance
rf_model_instance = rf_model.stages[-1]
feature_importances = rf_model_instance.featureImportances
print(f"\nTop 5 Feature Importances:")
for i, importance in enumerate(feature_importances.toArray()[:5]):
    print(f"Feature {i}: {importance:.4f}")

Training Random Forest model...

=== RANDOM FOREST MODEL EVALUATION ===
AUC-ROC: 0.6467
Accuracy: 0.9246
Weighted Precision: 0.8549
Weighted Recall: 0.9246
F1 Score: 0.8884

Top 5 Feature Importances:
Feature 0: 0.0995
Feature 1: 0.0421
Feature 2: 0.0834
Feature 3: 0.0606
Feature 4: 0.1025


Use Case 2 Revenue Prediction with Linear Regression


In [0]:
# Cell 3: Linear Regression for Monthly Spend Prediction
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Prepare data for regression
df_revenue = df.select(
    'Age', 'Total_Watch_Hours_90', 'Num_Sessions_Last_30',
    'Avg_Session_Min', 'Num_Ratings', 'Avg_Rating', 
    'Sentiment_Score', 'Monthly_Spend_USD',
    'Gender', 'Subscription_Plan', 'Region'
)

# Filter out null values and create label
df_revenue = df_revenue.filter(F.col('Monthly_Spend_USD').isNotNull())
df_revenue = df_revenue.withColumn('label', F.col('Monthly_Spend_USD'))

# Handle missing values
numeric_features = ['Age', 'Total_Watch_Hours_90', 'Num_Sessions_Last_30',
                   'Avg_Session_Min', 'Num_Ratings', 'Avg_Rating', 'Sentiment_Score']
for col in numeric_features:
    df_revenue = df_revenue.fillna({col: 0})

# String indexing
gender_idx = StringIndexer(inputCol="Gender", outputCol="Gender_Idx", handleInvalid="keep")
plan_idx = StringIndexer(inputCol="Subscription_Plan", outputCol="Plan_Idx", handleInvalid="keep")
region_idx = StringIndexer(inputCol="Region", outputCol="Region_Idx", handleInvalid="keep")

# Feature assembly
feature_cols = numeric_features + ['Gender_Idx', 'Plan_Idx', 'Region_Idx']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Linear Regression
lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.8
)

# Pipeline
lr_pipeline = Pipeline(stages=[gender_idx, plan_idx, region_idx, assembler, lr])

# Split data
train_data, test_data = df_revenue.randomSplit([0.8, 0.2], seed=42)

# Train model
print("Training Linear Regression model...")
lr_model = lr_pipeline.fit(train_data)

# Predictions
lr_predictions = lr_model.transform(test_data)

# Evaluation
print("\n=== LINEAR REGRESSION MODEL EVALUATION ===")

# RMSE
rmse_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
rmse = rmse_evaluator.evaluate(lr_predictions)
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

# MAE
mae_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
mae = mae_evaluator.evaluate(lr_predictions)
print(f"Mean Absolute Error (MAE): {mae:.4f}")

# R2
r2_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")
r2 = r2_evaluator.evaluate(lr_predictions)
print(f"R-squared (R2): {r2:.4f}")

# Get coefficients
# Get the LinearRegression model from the pipeline
lr_model_instance = lr_model.stages[-1]

print(f"\nLinear Regression Coefficients:")
print(f"Intercept: {lr_model_instance.intercept:.4f}")

# Get the coefficients (this is a Spark Vector)
coefficients = lr_model_instance.coefficients

# --- CORRECTION ---
# 1. Use the correct variable name 'coefficients'
# 2. Convert it to a Python list
coefficients_list = coefficients.toArray().tolist()
# --- END CORRECTION ---

# Now, use the new 'coefficients_list' for len() and slicing
print(f"Number of features: {len(coefficients_list)}")
print(f"Top 5 coefficients: {coefficients_list[-5:]}") 

# Show sample predictions
print(f"\nSample Predictions vs Actual:")
lr_predictions.select("label", "prediction").show(10)

# Show sample predictions
print("\nSample Predictions vs Actual:")
lr_predictions.select("label", "prediction").show(10)

Training Linear Regression model...

=== LINEAR REGRESSION MODEL EVALUATION ===
Root Mean Squared Error (RMSE): 3.8205
Mean Absolute Error (MAE): 3.1728
R-squared (R2): 0.0927

Linear Regression Coefficients:
Intercept: 2.7953
Number of features: 10
Top 5 coefficients: [0.0, 0.0, -0.1432423561967874, 0.0, 0.0]

Sample Predictions vs Actual:
+-----+------------------+
|label|        prediction|
+-----+------------------+
| 3.99| 4.166633220785098|
| 7.99|5.1414573469486236|
| 3.99| 4.805396966593255|
|  0.0| 4.594718722850674|
| 7.99| 4.624137594900775|
|  0.0|4.2418107410269785|
| 7.99|  4.87620965854356|
| 3.99| 5.878756091093667|
| 3.99| 5.594322403171629|
|  0.0|6.1870567169595585|
+-----+------------------+
only showing top 10 rows

Sample Predictions vs Actual:
+-----+------------------+
|label|        prediction|
+-----+------------------+
| 3.99| 4.166633220785098|
| 7.99|5.1414573469486236|
| 3.99| 4.805396966593255|
|  0.0| 4.594718722850674|
| 7.99| 4.624137594900775|
|  0.0|

Use Case 3 User Segmentation with Decision Trees

In [0]:
# Cell 4: Decision Tree Classifier for User Segmentation
from pyspark.ml.classification import DecisionTreeClassifier

# Create user segments based on behavior
df_segments = df.withColumn(
    'user_segment',
    F.when((F.col('Monthly_Spend_USD') > 15) & (F.col('Total_Watch_Hours_90') > 60), 0)  # Premium
     .when((F.col('Monthly_Spend_USD') > 10) & (F.col('Total_Watch_Hours_90') > 30), 1)  # Regular
     .when((F.col('Monthly_Spend_USD') <= 10) & (F.col('Total_Watch_Hours_90') > 30), 2)  # Engaged Low Spender
     .otherwise(3)  # Low Value
)

# Select features
features = ['Age', 'Total_Watch_Hours_90', 'Num_Sessions_Last_30', 
           'Avg_Session_Min', 'Sentiment_Score', 'Ad_Click_Rate']

df_dt = df_segments.select(features + ['user_segment', 'Gender', 'Preferred_Device'])
df_dt = df_dt.filter(F.col('user_segment').isNotNull())
df_dt = df_dt.withColumn('label', F.col('user_segment'))

# Handle missing values
for col in features:
    df_dt = df_dt.fillna({col: 0})

# String indexing
gender_idx = StringIndexer(inputCol="Gender", outputCol="Gender_Idx", handleInvalid="keep")
device_idx = StringIndexer(inputCol="Preferred_Device", outputCol="Device_Idx", handleInvalid="keep")

# Feature assembly
all_features = features + ['Gender_Idx', 'Device_Idx']
assembler = VectorAssembler(inputCols=all_features, outputCol="features")

# Decision Tree
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=5,
    maxBins=32,
    seed=42
)

# Pipeline
dt_pipeline = Pipeline(stages=[gender_idx, device_idx, assembler, dt])

# Split data
train_data, test_data = df_dt.randomSplit([0.8, 0.2], seed=42)

# Train model
print("Training Decision Tree model...")
dt_model = dt_pipeline.fit(train_data)

# Predictions
dt_predictions = dt_model.transform(test_data)

# Evaluation
print("\n=== DECISION TREE MODEL EVALUATION ===")

# Accuracy
accuracy_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
accuracy = accuracy_eval.evaluate(dt_predictions)
print(f"Accuracy: {accuracy:.4f}")

# Precision
precision_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision")
precision = precision_eval.evaluate(dt_predictions)
print(f"Weighted Precision: {precision:.4f}")

# Recall
recall_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall")
recall = recall_eval.evaluate(dt_predictions)
print(f"Weighted Recall: {recall:.4f}")

# F1
f1_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
f1 = f1_eval.evaluate(dt_predictions)
print(f"F1 Score: {f1:.4f}")

# Feature importance
dt_model_instance = dt_model.stages[-1]
print(f"\nDecision Tree Depth: {dt_model_instance.depth}")
print(f"Number of Nodes: {dt_model_instance.numNodes}")

# Confusion Matrix
confusion_matrix = dt_predictions.groupBy("label", "prediction").count()
print("\nConfusion Matrix:")
confusion_matrix.show()

Training Decision Tree model...

=== DECISION TREE MODEL EVALUATION ===
Accuracy: 0.8945
Weighted Precision: 0.8090
Weighted Recall: 0.8945
F1 Score: 0.8485

Decision Tree Depth: 5
Number of Nodes: 31

Confusion Matrix:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       2.0|   20|
|    2|       3.0|    1|
|    2|       2.0|  108|
|    3|       3.0|   70|
+-----+----------+-----+



Use Case 4 Gradient Boosted Trees for Watch Time Prediction

In [0]:
# Cell 5: GBT Regressor for Watch Time Prediction
from pyspark.ml.regression import GBTRegressor

# Prepare data for watch time prediction
df_watch = df.select(
    'Age', 'Monthly_Spend_USD', 'Num_Sessions_Last_30',
    'Avg_Session_Min', 'Sentiment_Score', 'Total_Watch_Hours_90',
    'Gender', 'Subscription_Plan', 'Preferred_Language'
)

df_watch = df_watch.filter(F.col('Total_Watch_Hours_90').isNotNull())
df_watch = df_watch.withColumn('label', F.col('Total_Watch_Hours_90'))

# Handle missing values
numeric_cols = ['Age', 'Monthly_Spend_USD', 'Num_Sessions_Last_30', 
                'Avg_Session_Min', 'Sentiment_Score']
for col in numeric_cols:
    df_watch = df_watch.fillna({col: 0})

# String indexing
gender_idx = StringIndexer(inputCol="Gender", outputCol="Gender_Idx", handleInvalid="keep")
plan_idx = StringIndexer(inputCol="Subscription_Plan", outputCol="Plan_Idx", handleInvalid="keep")
lang_idx = StringIndexer(inputCol="Preferred_Language", outputCol="Language_Idx", handleInvalid="keep")

# Feature assembly
feature_cols = numeric_cols + ['Gender_Idx', 'Plan_Idx', 'Language_Idx']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# GBT Regressor
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    maxDepth=5,
    seed=42
)

# Pipeline
gbt_pipeline = Pipeline(stages=[gender_idx, plan_idx, lang_idx, assembler, gbt])

# Split data
train_data, test_data = df_watch.randomSplit([0.8, 0.2], seed=42)

# Train model
print("Training Gradient Boosted Trees model...")
gbt_model = gbt_pipeline.fit(train_data)

# Predictions
gbt_predictions = gbt_model.transform(test_data)

# Evaluation
print("\n=== GRADIENT BOOSTED TREES MODEL EVALUATION ===")

# RMSE
rmse_eval = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
rmse = rmse_eval.evaluate(gbt_predictions)
print(f"RMSE: {rmse:.4f}")

# MAE
mae_eval = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
mae = mae_eval.evaluate(gbt_predictions)
print(f"MAE: {mae:.4f}")

# R2
r2_eval = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")
r2 = r2_eval.evaluate(gbt_predictions)
print(f"R-squared: {r2:.4f}")

# Feature importance
gbt_model_instance = gbt_model.stages[-1]
print(f"\nGBT Feature Importances:")
for i, importance in enumerate(gbt_model_instance.featureImportances.toArray()[:5]):
    print(f"Feature {i}: {importance:.4f}")

Training Gradient Boosted Trees model...

=== GRADIENT BOOSTED TREES MODEL EVALUATION ===
RMSE: 51.2321
MAE: 39.4216
R-squared: -0.2859

GBT Feature Importances:
Feature 0: 0.1537
Feature 1: 0.0597
Feature 2: 0.1378
Feature 3: 0.2099
Feature 4: 0.2352


Use Case 5 Model Comparison Dashboard


In [0]:
import pandas as pd

# --- Fill in your metrics from previous cells ---
# (I've used the values from your Linear Regression screenshot)
# (The other values are placeholders - replace them with your variables)

# Linear Regression metrics (from your previous screenshot)
lr_rmse = 3.8205
lr_mae = 3.1728
lr_r2 = 0.0927

# Random Forest metrics (replace with your RF model's results)
rf_rmse = 2.95  # Example: rf_metrics['rmse']
rf_mae = 2.41   # Example: rf_metrics['mae']
rf_r2 = 0.45    # Example: rf_metrics['r2']

# ALS metrics (replace with your best ALS model's RMSE)
# This would be the 'best_validation_rmse' from our MLflow run
als_best_rmse = 0.87 # Example: best_rmse 
# ---

# Create a summary of all model performances
model_results = {
    'Model': [
        'Linear Regression', 
        'Random Forest', 
        'ALS (Recommender)'
    ],
    'RMSE': [
        lr_rmse, 
        rf_rmse, 
        als_best_rmse  # ALS is only evaluated on RMSE here
    ],
    'MAE': [
        lr_mae, 
        rf_mae, 
        None  # MAE was not a primary metric for our ALS run
    ],
    'R-squared (R2)': [
        lr_r2, 
        rf_r2, 
        None  # R-squared is not applicable to matrix factorization
    ]
}

# Create a Pandas DataFrame
results_df = pd.DataFrame(model_results)

# Set 'Model' as the index for better readability
results_df = results_df.set_index('Model')

# Display the final comparison table
print("=== Model Performance Comparison ===")
display(results_df)

=== Model Performance Comparison ===


RMSE,MAE,R-squared (R2)
3.8205,3.1728,0.0927
2.95,2.41,0.45
0.87,null,null
